<a href="https://colab.research.google.com/github/harish-karthikeyan/BigData/blob/main/diabetesDay9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import NaiveBayes
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

In [2]:
spark = SparkSession.builder.appName("DiabetesNaiveBayes").getOrCreate()
df = spark.read.csv("/content/diabetes.csv", header=True, inferSchema=True)
df.printSchema()

root
 |-- Pregnancies: integer (nullable = true)
 |-- Glucose: integer (nullable = true)
 |-- BloodPressure: integer (nullable = true)
 |-- SkinThickness: integer (nullable = true)
 |-- Insulin: integer (nullable = true)
 |-- BMI: double (nullable = true)
 |-- DiabetesPedigreeFunction: double (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Outcome: integer (nullable = true)



In [3]:
feature_cols = ["Pregnancies","Glucose","BloodPressure","SkinThickness","Insulin","BMI","DiabetesPedigreeFunction","Age"]
target_col = "Outcome"

In [4]:
assembler = VectorAssembler(inputCols=feature_cols, outputCol="raw_features")
data_assembled = assembler.transform(df)

In [5]:
scaler = StandardScaler(inputCol="raw_features", outputCol="features", withStd=True, withMean=True)
scaler_model = scaler.fit(data_assembled)
scaled_data = scaler_model.transform(data_assembled)

In [6]:
train_data, test_data = scaled_data.randomSplit([0.8, 0.2], seed=42)

In [7]:
nb = NaiveBayes(featuresCol="features", labelCol=target_col, modelType="gaussian")
nb_model = nb.fit(train_data)

In [8]:
predictions = nb_model.transform(test_data)
predictions.select("Outcome", "prediction", "probability").show(10, truncate=False)

+-------+----------+-----------------------------------------+
|Outcome|prediction|probability                              |
+-------+----------+-----------------------------------------+
|0      |0.0       |[0.8970444334308734,0.10295556656912667] |
|0      |0.0       |[0.9502906010054543,0.04970939899454577] |
|0      |0.0       |[0.8987722629406428,0.10122773705935721] |
|0      |0.0       |[0.98336957891023,0.016630421089769837]  |
|0      |0.0       |[0.984763943245035,0.01523605675496509]  |
|0      |0.0       |[0.9871874807767563,0.012812519223243668]|
|0      |0.0       |[0.9985452517353335,0.001454748264666535]|
|0      |0.0       |[0.8983776242279005,0.10162237577209961] |
|0      |0.0       |[0.9566818693014127,0.043318130698587205]|
|1      |0.0       |[0.9554429436138988,0.04455705638610129] |
+-------+----------+-----------------------------------------+
only showing top 10 rows


In [9]:
accuracy_evaluator = MulticlassClassificationEvaluator(labelCol=target_col, predictionCol="prediction", metricName="accuracy")
accuracy = accuracy_evaluator.evaluate(predictions)

In [10]:
auc_evaluator = BinaryClassificationEvaluator(labelCol=target_col, rawPredictionCol="rawPrediction", metricName="areaUnderROC")
auc = auc_evaluator.evaluate(predictions)

In [11]:
print("=" * 40)
print(f"Test Accuracy: {accuracy * 100:.2f}%")
print(f"ROC AUC Score: {auc:.4f}")
print("=" * 40)

Test Accuracy: 73.17%
ROC AUC Score: 0.6093


In [12]:
spark.stop()